In [129]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [130]:
keras.mixed_precision.set_global_policy('mixed_bfloat16')

In [131]:
import pickle
import tarfile
import datetime
import numpy as np
import pandas as pd
import urllib.request
import sklearn.metrics
import matplotlib.pyplot as plt
import zipfile
from joblib import Parallel, delayed
import albumentations as A

from pathlib import Path
import cv2
from typing import Literal

In [132]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: True


In [133]:
BATCH_SIZE = 256
LOGS_DIR = '../logs'
DATA_DIR = '../data'

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [134]:
def download_data():
    filepath = os.path.join(DATA_DIR, 'tiny-imagenet-200.zip')
    if not os.path.exists('tiny-imagenet-200'):
        print("Donwloading...")
        urllib.request.urlretrieve('http://cs231n.stanford.edu/tiny-imagenet-200.zip', filepath)

        print("Extracting...")
        file = zipfile.ZipFile(filepath, 'r')
        file.extractall(DATA_DIR)

In [135]:
# download_data()

In [136]:
class TinyImageNetDataset(keras.utils.PyDataset):

    def __init__(
        self,
        path: str,
        mode: Literal["train", "val"],
        *,
        batch_size=32,
        shuffle=True,
        seed=None,
        preload=True,
        augment: A.Compose | None = None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.shuffle = shuffle
        self.prefetch = preload
        self.mode = mode
        self.augment = augment

        self.base_path = Path(path)

        if mode == "train":
            self.read_train(self.base_path)
        else:
            self.read_val(self.base_path)

        self.on_epoch_end()

    def read_train(self, base_path: Path):
        train_path = base_path / "train"

        classes, class_mapping = self.get_class_mapping(train_path)

        self.image_paths: list[str] = []
        labels: list[int] = []

        for img_class in classes:
            img_dir = train_path / img_class / "images"

            for img_file in img_dir.glob("*.JPEG"):
                self.image_paths.append(str(img_file))
                labels.append(class_mapping[img_class])

        self.indices = np.arange(len(self.image_paths))
        self.labels = np.array(labels)

        if self.prefetch:
            self.images_data = np.array(
                Parallel(n_jobs=-1)(
                    delayed(cv2.imread)(path) for path in self.image_paths
                )
            )

    def read_val(self, base_path: Path):
        _, mapping = self.get_class_mapping(base_path / "train")
        val_path = base_path / "val"

        df = pd.read_csv(val_path / "val_annotations.txt", delimiter="\t", header=None)
        image_dir = val_path / "images"

        self.image_paths = []
        labels = []

        for index, row in df.iterrows():
            self.image_paths.append(str(image_dir / row[0]))
            labels.append(mapping[row[1]])

        self.labels = np.array(labels)
        self.indices = np.arange(len(self.image_paths))
        if self.prefetch:
            self.images_data = np.array(
                Parallel(n_jobs=-1)(
                    delayed(cv2.imread)(path) for path in self.image_paths
                )
            )

    def get_class_mapping(self, train_path: Path) -> tuple[list[str], dict[str, int]]:
        classes: list[str] = []
        for item in train_path.iterdir():
            if item.is_dir():
                classes.append(item.name)

        return classes, {
            class_name: idx for idx, class_name in enumerate(sorted(classes))
        }

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(start + self.batch_size, len(self.image_paths))
        batch_indices = self.indices[start:end]

        if self.prefetch:
            images = self.images_data[batch_indices]
        else:
            images = np.array(
                Parallel(n_jobs=-1)(
                    delayed(cv2.imread)(self.image_paths[i]) for i in batch_indices
                )
            )

        if self.mode == "train" and self.augment:
            images = np.array([self.augment(image=img)["image"] for img in images])

        return (
            images,
            self.labels[batch_indices],
        )

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

In [137]:
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), rotate=10, translate_percent=0.05, p=0.4),
    A.RandomBrightnessContrast(p=0.2)
], p=1.0)

In [138]:
imagenet_path = "../data/tiny-imagenet-200"

In [139]:
train_dataset = TinyImageNetDataset(
    imagenet_path,
    "train",
    batch_size=BATCH_SIZE,
    workers=1,
    use_multiprocessing=False,
    augment=augment
)

In [140]:
values, counts = np.unique_counts(train_dataset.labels)

In [141]:
np.unique_counts(counts)

UniqueCountsResult(values=array([500]), counts=array([200]))

In [142]:
val_dataset = TinyImageNetDataset(
    imagenet_path,
    "val",
    shuffle=False,
    batch_size=BATCH_SIZE,
    workers=1,
    use_multiprocessing=False,
)

In [143]:
input_shape = (64, 64, 3)
num_classes = 200

In [145]:
y = inputs_y = keras.Input(shape=input_shape)

y = keras.layers.Rescaling(1./255)(y)

y = keras.layers.Conv2D(32, 3, padding="same")(y)
y = keras.layers.BatchNormalization()(y)
y = keras.layers.ReLU()(y)

y = keras.layers.GlobalAveragePooling2D()(y)
y = keras.layers.Dense(64)(y)
y = keras.layers.BatchNormalization()(y)
y = keras.layers.ReLU()(y)
y = keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(y)

In [146]:
model = keras.models.Model(inputs=inputs_y, outputs=y)

In [148]:
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_6 (Rescaling)         │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_36 (ReLU)                 │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 32)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_37          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_37 (ReLU)                 │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 200)            │        13,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,392 (64.03 KB)

 Trainable params: 16,200 (63.28 KB)

 Non-trainable params: 192 (768.00 B)

In [149]:
optimizer = keras.optimizers.Adam(
    learning_rate=0.0005,
    weight_decay=1e-4
)

In [150]:
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [151]:
logdir = os.path.join(LOGS_DIR, "tiny_imagenet", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

In [152]:
class PyTorchTensorBoard(keras.callbacks.Callback):
    def __init__(self, log_dir):
        super().__init__()
        from torch.utils.tensorboard import SummaryWriter
        self.writer = SummaryWriter(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        if logs:
            for key, value in logs.items():
                self.writer.add_scalar(key, value, epoch)


In [153]:
import gc

class Stabilizer(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [154]:
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(logdir, 'model.keras'),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    PyTorchTensorBoard(logdir),
    Stabilizer()
]

In [155]:
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=40,
    callbacks=callbacks,
)

Epoch 1/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.0181 - loss: 5.2042
Epoch 1: val_accuracy improved from None to 0.01620, saving model to ../logs\tiny_imagenet\20260522-143625\model.keras

Epoch 1: finished saving model to ../logs\tiny_imagenet\20260522-143625\model.keras
391/391 ━━━━━━━━━━━━━━━━━━━━ 30s 77ms/step - accuracy: 0.0276 - loss: 5.0851 - val_accuracy: 0.0162 - val_loss: 5.1432 - learning_rate: 5.0000e-04
Epoch 2/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.0467 - loss: 4.8713
Epoch 2: val_accuracy improved from 0.01620 to 0.03980, saving model to ../logs\tiny_imagenet\20260522-143625\model.keras

Epoch 2: finished saving model to ../logs\tiny_imagenet\20260522-143625\model.keras
391/391 ━━━━━━━━━━━━━━━━━━━━ 38s 98ms/step - accuracy: 0.0509 - loss: 4.8286 - val_accuracy: 0.0398 - val_loss: 4.8756 - learning_rate: 5.0000e-04
Epoch 3/40
391/391 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 0.0634 - loss: 4.6905
Epoch 3: val_accuracy improved fr

KeyboardInterrupt: 

In [156]:
y_true = val_dataset.labels
y_pred = model.predict(val_dataset).argmax(axis=-1)

40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


In [ ]:
# _, ax = plt.subplots(figsize=(75, 75))
# sklearn.metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False, display_labels=names, xticks_rotation=90)

# plt.tight_layout()

In [157]:
np.sum(y_true == y_pred) / len(y_true)

np.float64(0.148)

In [158]:
_, mapping = train_dataset.get_class_mapping(train_dataset.base_path / "train")
names_df = pd.read_csv(train_dataset.base_path / "words.txt", sep="\t", header=None)
names = [""] * len(mapping)
for class_name, i in mapping.items():
    names[i] = names_df.loc[names_df.iloc[:, 0] == class_name][1].iloc[0]

In [159]:
import plotly.express as px
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(y_true, y_pred)

fig = px.imshow(
    cm,
    x=names,
    y=names,
    color_continuous_scale='Viridis',
    aspect='equal'
)

fig.update_layout(
    title='Confusion Matrix (200x200)',
    xaxis_title='Predicted',
    yaxis_title='True',
    hovermode='closest',
    width=1200,
    height=1200,
    xaxis={'tickfont': {'size': 8}},
    yaxis={'tickfont': {'size': 8}}
)

fig.write_html(os.path.join(logdir, 'matrix-10ep.html'), auto_open=True)